# AI Smart Shopping Basket Optimizer

## Notebook 04: Product Recommendation System

### Objective

The objective of this notebook is to build a product recommendation system by combining market basket analysis results with the BigBasket product dataset. The system will recommend relevant products along with their brand, price, rating, and discount information.

In [2]:
import pandas as pd
import numpy as np

In [3]:
products = pd.read_csv(
    r"C:\Users\sampa\Desktop\AI-Smart-Shopping-Basket-Optimizer\data\processed\bigbasket_cleaned.csv"
)

products.head()

,product,category,sub_category,brand,sale_price,market_price,type,rating,description
0,Garlic Oil - Vegetarian Capsule 500 mg,Beauty & Hygiene,Hair Care,Sri Sri Ayurveda,220.0,220.0,Hair Oil & Serum,4.1,This Product contains Garlic Oil that is known...
1,Water Bottle - Orange,"Kitchen, Garden & Pets",Storage & Accessories,Mastercook,180.0,180.0,Water & Fridge Bottles,2.3,"Each product is microwave safe (without lid), ..."
2,"Brass Angle Deep - Plain, No.2",Cleaning & Household,Pooja Needs,Trm,119.0,250.0,Lamp & Lamp Oil,3.4,"A perfect gift for all occasions, be it your m..."
3,Cereal Flip Lid Container/Storage Jar - Assort...,Cleaning & Household,Bins & Bathroom Ware,Nakoda,149.0,176.0,"Laundry, Storage Baskets",3.7,Multipurpose container with an attractive desi...
4,Creme Soft Soap - For Hands & Body,Beauty & Hygiene,Bath & Hand Wash,Nivea,162.0,162.0,Bathing Bars & Soaps,4.4,Nivea Creme Soft Soap gives your skin the best...


In [4]:
products.columns

Index(['product', 'category', 'sub_category', 'brand', 'sale_price',
       'market_price', 'type', 'rating', 'description'],
      dtype='object')

In [5]:
def search_product(keyword):
    result = products[
        products["product"].str.contains(
            keyword,
            case=False,
            na=False
        )
    ]

    return result[
        [
            "product",
            "brand",
            "sale_price",
            "market_price",
            "rating"
        ]
    ]

In [9]:
groceries = pd.read_csv(
    r"C:\Users\sampa\Desktop\AI-Smart-Shopping-Basket-Optimizer\data\raw\Groceries_dataset.csv"
)

groceries.head()

,Member_number,Date,itemDescription
0,1808,21-07-2015,tropical fruit
1,2552,05-01-2015,whole milk
2,2300,19-09-2015,pip fruit
3,1187,12-12-2015,other vegetables
4,3037,01-02-2015,whole milk


In [12]:
groceries = groceries.drop_duplicates()

groceries["Transaction"] = (
    groceries["Member_number"].astype(str)
    + "_"
    + groceries["Date"]
)

In [13]:
basket = (
    groceries.groupby(["Transaction", "itemDescription"])["itemDescription"]
    .count()
    .unstack()
    .fillna(0)
)

basket = basket.astype(bool)

In [14]:
from mlxtend.frequent_patterns import apriori, association_rules

In [15]:
frequent_items = apriori(
    basket,
    min_support=0.005,
    use_colnames=True
)

In [16]:
rules = association_rules(
    frequent_items,
    metric="confidence",
    min_threshold=0.1
)

In [6]:
search_product("milk")

,product,brand,sale_price,market_price,rating
92,Topp Up Milk - Elaichi,Gowardhan,80.01,90.0,5.0
215,Wonderz Milk Shakes - Classic Vanilla,Sunfeast,29.75,35.0,4.5
306,Milk Boiler - Aluminium,Le Kaviraj,549.00,732.0,2.5
318,Milk Delights Face Wash With Honey For Dry Skin,Nivea,86.40,90.0,4.2
357,"Cereal - Ragi, Rice & Mango With Milk",Slurrp Farm,300.00,300.0,4.0
...,...,...,...,...,...
27190,Flavoured Milk - Strawberry,Godrej Jersey,30.00,30.0,4.5
27215,"Belgian Milk Chocolate, Velvety & Smooth",Amul,140.00,150.0,4.3
27275,Bathing Soap - Jasmine & Milk Cream,Godrej No.1,84.00,84.0,4.3
27282,Banana Milk Shake,Amul,91.00,100.0,4.3


In [7]:
def recommend_products(product_name):
    matched_rules = rules[
        rules["antecedents"].apply(
            lambda x: product_name.lower() in [item.lower() for item in x]
        )
    ]

    if matched_rules.empty:
        return "No recommendations found."

    recommendations = []

    for items in matched_rules["consequents"]:
        for item in items:
            recommendations.append(item)

    return recommendations

In [22]:
recommend_products("yogurt")

['whole milk']

In [25]:
def get_product_details(product_list):

    result = products[
        products["product"].str.contains(
            "|".join(product_list),
            case=False,
            na=False
        )
    ]

    return result[
        [
            "product",
            "brand",
            "sale_price",
            "market_price",
            "rating"
        ]
    ]

In [26]:
recommended = recommend_products("yogurt")

recommended

['whole milk']

In [27]:
get_product_details(recommended)

,product,brand,sale_price,market_price,rating
17299,Whole Milk Powder,Puramate,76.5,85.0,3.8


In [28]:
def recommend_with_details(product_name):
    recommended = recommend_products(product_name)

    if recommended == "No recommendations found.":
        return recommended

    return get_product_details(recommended)

In [29]:
recommend_with_details("yogurt")

,product,brand,sale_price,market_price,rating
17299,Whole Milk Powder,Puramate,76.5,85.0,3.8
